# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Runs the CIPHER metacognition benchmark against as many frontier LLMs as possible
and registers the results on Kaggle Benchmarks.

### Kaggle Dataset required
Upload a dataset named **`cipher-benchmark`** containing:
```
cipher/          ← the Python package
scripts/         ← evaluate.py etc.
data/instances.jsonl   ← 1 000 pre-generated instances (seed=2026)
README.md
```
Then attach it to this notebook at `/kaggle/input/cipher-benchmark/`.

### Scoring dimensions (all normalized to [0, 1])
| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35 % | Final plan quality vs oracle beam search |
| Calibration | 25 % | Brier score on stated self-knowledge |
| Attention | 20 % | Rank correlation on unknown importance |
| Executive | 20 % | Plan structure, named risks, alternative plans |

### API keys (Kaggle Secrets)
Add secrets under **Add-ons → Secrets** before running:
- `MODEL_PROXY_API_KEY` — Kaggle-provisioned quota ($50/day), covers all Gemini models
- `ANTHROPIC_API_KEY` — optional, for Claude models
- `OPENAI_API_KEY` — optional, for GPT-4o family
- `HF_TOKEN` — optional, for open-source models via HuggingFace Inference

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys

def _pip(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *packages],
        check=True,
    )

_pip("kaggle-benchmarks")
_pip("google-genai")        # unified Google GenAI SDK (used by kbench too)
_pip("anthropic")           # Claude
_pip("openai")              # GPT-4o family
_pip("huggingface_hub")     # open-source models via HF Inference Providers
_pip("tqdm", "pandas", "matplotlib")
print("Dependencies installed.")

In [ ]:
# ── 2. Imports & path setup ───────────────────────────────────────────────────
import os, sys, json, random, time, datetime
from typing import Any, Dict, List, Optional

# On Kaggle the dataset is at /kaggle/input/<dataset-slug>/
# Locally fall back to the repo root (one level above this notebook).
_KAGGLE_ROOT = "/kaggle/input/cipher-benchmark"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())  # works in Jupyter; no __file__ in notebooks
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _candidate in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_candidate, "cipher")):
        CIPHER_ROOT = _candidate
        break

if CIPHER_ROOT is None:
    raise RuntimeError(
        "cipher/ package not found.  Attach the 'cipher-benchmark' dataset "
        "or run from inside the repo directory."
    )

if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)

print(f"cipher root: {CIPHER_ROOT}")

from cipher import build_prompt
from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect, Action
from cipher.simulator import run_plan
from cipher.schema import validate_response
from cipher.scorer import score_response
from cipher.optimal import oracle_score

print("cipher imports OK.")

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────

# ── Data path ────────────────────────────────────────────────────────────────
_data_candidates = [
    os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
    os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
]
DATA_PATH = next((p for p in _data_candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "data/instances.jsonl not found. Generate it first:\n"
        "  python scripts/generate_dataset.py --n 1000 --out data/instances.jsonl --seed 2026 --oracle"
    )

# ── Eval budget ───────────────────────────────────────────────────────────────
# 50 instances gives a stable signal and keeps API cost low.
# Set to None to run all 1 000 for the final Kaggle Benchmark submission.
EVAL_LIMIT   = 50
KBENCH_LIMIT = EVAL_LIMIT  # set to None for the full 1 000-instance kbench run

# ── API keys (Kaggle Secrets or env vars) ─────────────────────────────────────
def _secret(name: str) -> str:
    if val := os.environ.get(name, ""):
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

MODEL_PROXY_API_KEY = _secret("MODEL_PROXY_API_KEY")  # Kaggle-provisioned quota
ANTHROPIC_API_KEY   = _secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY      = _secret("OPENAI_API_KEY")
HF_TOKEN            = _secret("HF_TOKEN")

print(f"DATA_PATH           : {DATA_PATH}")
print(f"EVAL_LIMIT          : {EVAL_LIMIT}")
print(f"MODEL_PROXY_API_KEY : {'✓ set' if MODEL_PROXY_API_KEY else '✗ MISSING'}")
print(f"ANTHROPIC_API_KEY   : {'✓ set' if ANTHROPIC_API_KEY   else '✗ missing (Claude skipped)'}")
print(f"OPENAI_API_KEY      : {'✓ set' if OPENAI_API_KEY      else '✗ missing (OpenAI skipped)'}")
print(f"HF_TOKEN            : {'✓ set' if HF_TOKEN            else '✗ missing (HF models skipped)'}")

In [ ]:
# ── 4. Load instances ─────────────────────────────────────────────────────────

def _load_jsonl(path: str, limit: Optional[int] = None) -> List[Dict]:
    with open(path) as f:
        recs = [json.loads(line) for line in f if line.strip()]
    return recs[:limit] if limit else recs


def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    """Re-hydrate a JSONL record into a live Instance (mirrors scripts/evaluate.py)."""
    h = rec["hidden"]
    hrl = {entry["rule_name"]: set(entry["hidden"])
           for entry in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t = r["trigger"]
        e = r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(
                kind=t["kind"], i=t["i"], j=t.get("j", -1), k=t.get("k", 0),
                hidden_kind=("trigger_kind" in hides),
                hidden_k=("trigger_k" in hides),
            ),
            effect=Effect(
                kind=e["kind"], target=e["target"],
                delta=e.get("delta", 0), source=e.get("source", -1),
                hidden_kind=("effect_kind" in hides),
                hidden_delta=("effect_delta" in hides),
            ),
        ))
    initial = State(tuple(
        EntityState(phase=s["phase"], flux=s["flux"])
        for s in h["initial_state"]
    ))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[],
        hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )


ALL_RECORDS   = _load_jsonl(DATA_PATH)
EVAL_RECORDS  = _load_jsonl(DATA_PATH, limit=EVAL_LIMIT)
KBENCH_RECORDS= _load_jsonl(DATA_PATH, limit=KBENCH_LIMIT)

print(f"Total instances : {len(ALL_RECORDS)}")
print(f"Eval   per model: {len(EVAL_RECORDS)}")
print(f"kbench per model: {len(KBENCH_RECORDS)}")

In [ ]:
# ── 5. Shared scoring helper ──────────────────────────────────────────────────

def _extract_json(text: str) -> Dict[str, Any]:
    """Pull the outermost JSON object out of a model reply."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return {}
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return {}


def score_text_response(raw_text: str, rec: Dict[str, Any]) -> Dict[str, Any]:
    """Parse a model reply and return a score-breakdown dict."""
    _zero = dict(composite=0.0, objective=0.0, calibration=0.0,
                 attention=0.0, executive=0.0)
    raw = _extract_json(raw_text)
    if not raw:
        return {**_zero, "error": "no_json"}
    inst      = _instance_from_record(rec)
    resp      = validate_response(raw)
    best_obj  = rec["hidden"].get("oracle_best")
    worst_obj = rec["hidden"].get("oracle_worst")
    bd        = score_response(resp, inst, best_obj=best_obj, worst_obj=worst_obj)
    return bd.to_dict()

In [ ]:
# ── 6. Stub agents (no API key needed) ───────────────────────────────────────

def stub_noop(inst: Instance) -> Dict[str, Any]:
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": True, "confidence": 0.5}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": "wait"}],
        "self_judgment": {"robustness_score": 50, "risks_identified": [],
                          "alternative_if_unknown_X": {}},
    }


def stub_random(inst: Instance) -> Dict[str, Any]:
    rng   = random.Random(inst.seed ^ 0xDEADBEEF)
    n     = inst.world.initial.n
    kinds = ["pulse", "damp", "shift", "unshift", "observe", "wait"]
    act   = lambda: {"kind": rng.choice(kinds), "i": rng.randrange(n)}
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": rng.random() > 0.5, "confidence": rng.random()}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": list(inst.true_unknown_ranking[::-1]),
        "exploratory_actions": [act() for _ in range(rng.randint(0, 2))],
        "final_plan":          [act() for _ in range(rng.randint(1, 5))],
        "self_judgment": {
            "robustness_score": 40,
            "risks_identified": ["random baseline"],
            "alternative_if_unknown_X": {"unknown": "", "plan": [act()]},
        },
    }


def stub_greedy(inst: Instance) -> Dict[str, Any]:
    """Beam-search the *visible* world and commit naively — good objective,
    poor calibration (claims everything is known)."""
    _, best_plan = oracle_score(inst.world)
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": True, "confidence": 0.9}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": a.kind, "i": a.i, "j": a.j} for a in best_plan],
        "self_judgment": {
            "robustness_score": 70,
            "risks_identified": [],
            "alternative_if_unknown_X": {},
        },
    }


print("Stub agents: noop / random / greedy")

In [ ]:
# ── 7. Gemini agents via Google GenAI SDK ────────────────────────────────────
# MODEL_PROXY_API_KEY gives $50/day free access to all Gemini tiers on Kaggle.

def _make_gemini_agent(model_id: str):
    def _agent(inst: Instance) -> Dict[str, Any]:
        import google.genai as genai
        from google.genai import types as gtypes
        client = genai.Client(api_key=MODEL_PROXY_API_KEY)
        resp = client.models.generate_content(
            model=model_id,
            contents=build_prompt(inst),
            config=gtypes.GenerateContentConfig(
                max_output_tokens=2048,
                temperature=0.0,
            ),
        )
        return _extract_json(resp.text or "")
    _agent.__name__ = model_id
    return _agent


# Full list of Gemini models accessible through the Kaggle proxy
GEMINI_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-1.5-pro",
    "gemini-1.5-flash",
    "gemini-1.5-flash-8b",
]

gemini_agents = (
    {m: _make_gemini_agent(m) for m in GEMINI_MODELS}
    if MODEL_PROXY_API_KEY else {}
)
print(f"Gemini agents: {list(gemini_agents) or ['(none — MODEL_PROXY_API_KEY not set)']}")

In [ ]:
# ── 8. Claude agents via Anthropic API ───────────────────────────────────────

def _make_claude_agent(model_id: str):
    def _agent(inst: Instance) -> Dict[str, Any]:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        msg = client.messages.create(
            model=model_id, max_tokens=2048,
            messages=[{"role": "user", "content": build_prompt(inst)}],
        )
        text = "".join(
            b.text for b in msg.content if getattr(b, "type", "") == "text"
        )
        return _extract_json(text)
    _agent.__name__ = model_id
    return _agent


CLAUDE_MODELS = [
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001",
]

claude_agents = (
    {m: _make_claude_agent(m) for m in CLAUDE_MODELS}
    if ANTHROPIC_API_KEY else {}
)
print(f"Claude agents : {list(claude_agents) or ['(none — ANTHROPIC_API_KEY not set)']}")

In [ ]:
# ── 9. OpenAI agents ─────────────────────────────────────────────────────────

def _make_openai_agent(model_id: str):
    # o1/o3 family don't support temperature; gpt-4o does
    _reasoning = model_id.startswith(("o1", "o3"))
    def _agent(inst: Instance) -> Dict[str, Any]:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        kwargs: Dict[str, Any] = {
            "model": model_id,
            "max_completion_tokens": 2048,
            "messages": [{"role": "user", "content": build_prompt(inst)}],
        }
        if not _reasoning:
            kwargs["temperature"] = 0.0
        resp = client.chat.completions.create(**kwargs)
        return _extract_json(resp.choices[0].message.content or "")
    _agent.__name__ = model_id
    return _agent


OPENAI_MODELS = [
    "gpt-4o",
    "gpt-4o-mini",
    "o1-mini",
    "o3-mini",
]

openai_agents = (
    {m: _make_openai_agent(m) for m in OPENAI_MODELS}
    if OPENAI_API_KEY else {}
)
print(f"OpenAI agents : {list(openai_agents) or ['(none — OPENAI_API_KEY not set)']}")

In [ ]:
# ── 10. HuggingFace Inference Provider agents ─────────────────────────────────

def _make_hf_agent(model_id: str, provider: str = "auto"):
    _short = model_id.split("/")[-1]
    def _agent(inst: Instance) -> Dict[str, Any]:
        from huggingface_hub import InferenceClient
        client = InferenceClient(model=model_id, token=HF_TOKEN, provider=provider)
        resp = client.chat_completion(
            messages=[{"role": "user", "content": build_prompt(inst)}],
            max_tokens=2048,
            temperature=0.0,
        )
        return _extract_json(resp.choices[0].message.content or "")
    _agent.__name__ = _short
    return _agent


HF_MODELS = [
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-72B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
]

hf_agents = (
    {m.split("/")[-1]: _make_hf_agent(m) for m in HF_MODELS}
    if HF_TOKEN else {}
)
print(f"HF agents     : {list(hf_agents) or ['(none — HF_TOKEN not set)']}")

In [ ]:
# ── 11. Master registry ───────────────────────────────────────────────────────

STUB_AGENTS = {
    "stub-noop":   stub_noop,
    "stub-random": stub_random,
    "stub-greedy": stub_greedy,
}

ALL_AGENTS: Dict[str, Any] = {
    **STUB_AGENTS,
    **gemini_agents,
    **claude_agents,
    **openai_agents,
    **hf_agents,
}

print(f"{len(ALL_AGENTS)} agents registered:")
for k in ALL_AGENTS:
    print(f"  · {k}")

In [ ]:
# ── 12. Evaluation loop ───────────────────────────────────────────────────────

from tqdm.auto import tqdm

def evaluate_agent(
    name: str,
    agent_fn,
    records: List[Dict],
) -> Dict[str, Any]:
    """
    Run agent_fn over records.  API errors are caught and zero-scored so one
    bad model doesn't abort the whole sweep.
    """
    per_instance = []
    n_errors = 0

    for rec in tqdm(records, desc=f"{name[:40]:40s}", leave=False):
        inst = _instance_from_record(rec)
        try:
            raw  = agent_fn(inst)
            resp = validate_response(raw)
            bd   = score_response(
                resp, inst,
                best_obj=rec["hidden"].get("oracle_best"),
                worst_obj=rec["hidden"].get("oracle_worst"),
            )
            per_instance.append({"id": inst.id, "difficulty": inst.difficulty,
                                  **bd.to_dict()})
        except Exception as exc:
            n_errors += 1
            per_instance.append({"id": inst.id, "difficulty": inst.difficulty,
                                  "composite": 0.0, "objective": 0.0,
                                  "calibration": 0.0, "attention": 0.0,
                                  "executive": 0.0, "error": str(exc)})

    n   = len(per_instance)
    avg = lambda k: sum(r.get(k, 0.0) for r in per_instance) / max(n, 1)
    return {
        "agent":            name,
        "n":                n,
        "n_errors":         n_errors,
        "mean_composite":   avg("composite"),
        "mean_objective":   avg("objective"),
        "mean_calibration": avg("calibration"),
        "mean_attention":   avg("attention"),
        "mean_executive":   avg("executive"),
        "per_instance":     per_instance,
        "timestamp":        datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }


RESULTS: Dict[str, Dict] = {}

for _name, _fn in ALL_AGENTS.items():
    print(f"\nEvaluating {_name} …")
    _t0 = time.time()
    _r  = evaluate_agent(_name, _fn, EVAL_RECORDS)
    RESULTS[_name] = _r
    print(
        f"  composite={_r['mean_composite']:.3f}  "
        f"obj={_r['mean_objective']:.3f}  "
        f"cal={_r['mean_calibration']:.3f}  "
        f"att={_r['mean_attention']:.3f}  "
        f"exe={_r['mean_executive']:.3f}  "
        f"errors={_r['n_errors']}  ({time.time()-_t0:.1f}s)"
    )

print("\n✓ All evaluations complete.")

In [ ]:
# ── 13. Leaderboard table ─────────────────────────────────────────────────────

import pandas as pd

_lb_rows = [
    {
        "Model":       r["agent"],
        "Composite ↑": round(r["mean_composite"],   3),
        "Objective":   round(r["mean_objective"],    3),
        "Calibration": round(r["mean_calibration"],  3),
        "Attention":   round(r["mean_attention"],    3),
        "Executive":   round(r["mean_executive"],    3),
        "N":           r["n"],
        "Errors":      r["n_errors"],
    }
    for r in sorted(RESULTS.values(), key=lambda x: -x["mean_composite"])
]

df_lb = pd.DataFrame(_lb_rows)
display(df_lb)

In [ ]:
# ── 14. Visualisation ─────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np

models     = df_lb["Model"].tolist()
composites = df_lb["Composite ↑"].tolist()
DIMS       = ["Objective", "Calibration", "Attention", "Executive"]

fig_h = max(5, len(models) * 0.5)
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(17, fig_h))

# ── Left panel: composite score ──────────────────────────────────────────────
y    = np.arange(len(models))
bars = ax_left.barh(y, composites, color="steelblue", edgecolor="white", height=0.65)
ax_left.set_yticks(y)
ax_left.set_yticklabels(models, fontsize=9)
ax_left.set_xlabel("Composite score (0 – 1)")
ax_left.set_title("CIPHER Leaderboard — Composite", fontweight="bold")
ax_left.set_xlim(0, 1.05)
ax_left.axvline(0.5, color="red", lw=1, ls="--", alpha=0.5, label="pass threshold")
ax_left.legend(fontsize=8)
for bar, val in zip(bars, composites):
    ax_left.text(
        bar.get_width() + 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center", fontsize=8,
    )
ax_left.invert_yaxis()

# ── Right panel: sub-score breakdown ─────────────────────────────────────────
x     = np.arange(len(DIMS))
width = min(0.8 / max(len(models), 1), 0.12)
cmap  = plt.get_cmap("tab20")
for idx, model in enumerate(models):
    row    = df_lb[df_lb["Model"] == model].iloc[0]
    scores = [float(row[d]) for d in DIMS]
    offset = (idx - len(models) / 2 + 0.5) * width
    ax_right.bar(x + offset, scores, width,
                 label=model, color=cmap(idx / max(len(models), 1)), alpha=0.85)
ax_right.set_xticks(x)
ax_right.set_xticklabels(DIMS)
ax_right.set_ylim(0, 1.0)
ax_right.set_ylabel("Score")
ax_right.set_title("Sub-score Breakdown", fontweight="bold")
n_cols = max(1, len(models) // 8)
ax_right.legend(fontsize=7, loc="upper right", ncol=n_cols)

plt.tight_layout()
os.makedirs("/kaggle/working", exist_ok=True)
_chart_path = "/kaggle/working/cipher_leaderboard.png"
plt.savefig(_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {_chart_path}")

In [ ]:
# ── 15. Breakdown by difficulty ───────────────────────────────────────────────

diff_rows = []
for name, result in RESULTS.items():
    by_diff: Dict[str, List[float]] = {"easy": [], "medium": [], "hard": []}
    for r in result["per_instance"]:
        by_diff.get(r.get("difficulty", "medium"), []).append(r.get("composite", 0.0))
    diff_rows.append({
        "Model":  name,
        "Easy":   round(sum(by_diff["easy"])   / max(len(by_diff["easy"]),   1), 3),
        "Medium": round(sum(by_diff["medium"]) / max(len(by_diff["medium"]), 1), 3),
        "Hard":   round(sum(by_diff["hard"])   / max(len(by_diff["hard"]),   1), 3),
    })

df_diff = pd.DataFrame(
    sorted(diff_rows, key=lambda r: -(r["Easy"]+r["Medium"]+r["Hard"]))
)
print("Composite by difficulty:")
display(df_diff)

In [ ]:
# ── 16. Save full results JSON ────────────────────────────────────────────────

os.makedirs("/kaggle/working", exist_ok=True)
_out = "/kaggle/working/cipher_all_results.json"
with open(_out, "w") as _f:
    json.dump(
        {
            "leaderboard": _lb_rows,
            "by_difficulty": diff_rows,
            "per_model": {
                k: {
                    "summary":      {kk: vv for kk, vv in v.items() if kk != "per_instance"},
                    "per_instance": v["per_instance"],
                }
                for k, v in RESULTS.items()
            },
        },
        _f, indent=2,
    )
print(f"Full results → {_out}")

---
## Kaggle Benchmarks — Official Task Registration

The cells below define and run the **`cipher_metacog`** task via the
`kaggle-benchmarks` SDK so results are recorded on the Kaggle Benchmarks
platform.

- Each `cipher_task.run()` call = one instance × one model.
- A run **passes** when `composite ≥ 0.5`; the platform records the
  pass-rate across all runs for each model.
- After execution, go to **https://www.kaggle.com/benchmarks/tasks/new**
  to publish the task, then create a Benchmark collection and attach it
  to your competition writeup.

> **Note**: these cells require `MODEL_PROXY_API_KEY` to be set and must
> run inside a **Kaggle Benchmark Task notebook** to register with the backend.

In [ ]:
# ── 17. kbench imports & task definition ─────────────────────────────────────

import kaggle_benchmarks as kbench
from kaggle_benchmarks.actors.llms import GoogleGenAI
import google.genai as genai
from google.genai import types as gtypes


@kbench.task(name="cipher_metacog")
def cipher_task(
    llm,
    instance_id: str,
    prompt: str,
    record_json: str,
):
    """
    CIPHER — Calibrated Introspection via Partially Hidden Environment Rules.

    The LLM is shown a partial causal world (invented vocabulary, hidden rule
    components) and must return strict JSON covering:
      metacog_assessment · critical_unknowns_ranked · exploratory_actions
      final_plan · self_judgment

    Scored across four axes:
      Objective (35%) · Calibration (25%) · Attention (20%) · Executive (20%)

    A run passes when composite ≥ 0.5.
    """
    reply     = llm.prompt(prompt)
    rec       = json.loads(record_json)
    breakdown = score_text_response(str(reply), rec)
    composite = breakdown.get("composite", 0.0)

    kbench.assertions.assert_true(
        composite >= 0.5,
        f"[{instance_id}] composite {composite:.3f} < 0.5 — "
        f"obj={breakdown.get('objective',0):.3f} "
        f"cal={breakdown.get('calibration',0):.3f} "
        f"att={breakdown.get('attention',0):.3f} "
        f"exe={breakdown.get('executive',0):.3f}",
    )


print(f"cipher_task defined.  Will run {len(KBENCH_RECORDS)} instances per model.")

In [ ]:
# ── 18. Run kbench for every Gemini model ────────────────────────────────────

if not MODEL_PROXY_API_KEY:
    print("Skipping kbench runs — MODEL_PROXY_API_KEY not set.")
else:
    _genai_client = genai.Client(api_key=MODEL_PROXY_API_KEY)

    for _model_id in GEMINI_MODELS:
        print(f"\n── kbench · {_model_id} ({len(KBENCH_RECORDS)} instances) ──")
        _llm = GoogleGenAI(client=_genai_client, model=_model_id)

        for _rec in tqdm(KBENCH_RECORDS, desc=_model_id, leave=False):
            try:
                cipher_task.run(
                    llm=_llm,
                    instance_id=_rec["id"],
                    prompt=_rec["prompt"],
                    record_json=json.dumps(_rec),
                )
            except Exception as _exc:
                print(f"  ⚠ {_rec['id']}: {_exc}")

        print(f"  ✓ {_model_id} done")

    print("\n✓ All Gemini kbench runs complete.")

In [ ]:
# ── 19. Run kbench for stub baselines ────────────────────────────────────────
# Stubs need the Instance object, but the kbench LLM interface only sees the
# prompt string.  We build a record-indexed stub LLM that reconstructs the
# Instance from the record_json argument passed by cipher_task.
#
# Trick: cipher_task passes record_json as a separate parameter; we intercept
# it by temporarily monkey-patching cipher_task inside a thin wrapper.

class _StubLLM:
    """Wraps a stub agent so it looks like a kbench LLM actor.

    `prepare(rec)` must be called once per run (before llm.prompt()) so the
    stub has access to the Instance.  We achieve this by overriding cipher_task
    for stub runs (see _run_stubs below).
    """
    def __init__(self, stub_fn):
        self._fn  = stub_fn
        self._rec: Optional[Dict] = None

    def set_rec(self, rec: Dict) -> None:
        self._rec = rec

    def prompt(self, _prompt_text: str) -> str:
        if self._rec is None:
            return "{}"
        inst = _instance_from_record(self._rec)
        return json.dumps(self._fn(inst))


def _run_stub_kbench(stub_name: str, stub_fn) -> None:
    """Run stub through kbench by pre-loading the record into the stub LLM."""
    _llm = _StubLLM(stub_fn)
    print(f"\n── kbench · {stub_name} ({len(KBENCH_RECORDS)} instances) ──")

    for _rec in tqdm(KBENCH_RECORDS, desc=stub_name, leave=False):
        _llm.set_rec(_rec)          # give the stub the current record
        try:
            cipher_task.run(
                llm=_llm,
                instance_id=_rec["id"],
                prompt=_rec["prompt"],
                record_json=json.dumps(_rec),
            )
        except Exception:
            pass  # stubs legitimately fall below 0.5; that's expected

    print(f"  ✓ {stub_name} done")


for _sname, _sfn in STUB_AGENTS.items():
    _run_stub_kbench(_sname, _sfn)

print("\n✓ Stub kbench runs complete.")

In [ ]:
# ── 20. Final summary ─────────────────────────────────────────────────────────

print("=" * 72)
print("CIPHER — FINAL LEADERBOARD")
print("=" * 72)
print(df_lb.to_string(index=False))
print()
print("Next steps:")
print("  1. Set KBENCH_LIMIT = None and re-run cells 17–19 for all 1 000 instances.")
print("  2. Go to https://www.kaggle.com/benchmarks/tasks/new")
print("     → publish the 'cipher_metacog' task.")
print("  3. Create a Benchmark collection named 'CIPHER' and add the task.")
print("  4. Copy the benchmark URL into your competition writeup under Project Links.")